PRACTICAL CONTINUOUS ASSESSMENT 1

Data Warehousing Lab

AutoClassify: An Intelligent Dataset-Aware Framework for Automatic Selection of Classification Algorithms


0. Download the dataset from in built scikit-learn


In [51]:
import pandas as pd
from sklearn.datasets import load_breast_cancer

# Load built-in dataset
data = load_breast_cancer(as_frame=True)

# Get complete dataframe
df = data.frame

# Save as CSV
df.to_csv("breast_cancer.csv", index=False)

print("CSV file created successfully!")
print("Shape:", df.shape)
print(df.head())

CSV file created successfully!
Shape: (569, 31)
   mean radius  mean texture  mean perimeter  mean area  mean smoothness  \
0        17.99         10.38          122.80     1001.0          0.11840   
1        20.57         17.77          132.90     1326.0          0.08474   
2        19.69         21.25          130.00     1203.0          0.10960   
3        11.42         20.38           77.58      386.1          0.14250   
4        20.29         14.34          135.10     1297.0          0.10030   

   mean compactness  mean concavity  mean concave points  mean symmetry  \
0           0.27760          0.3001              0.14710         0.2419   
1           0.07864          0.0869              0.07017         0.1812   
2           0.15990          0.1974              0.12790         0.2069   
3           0.28390          0.2414              0.10520         0.2597   
4           0.13280          0.1980              0.10430         0.1809   

   mean fractal dimension  ...  worst textur

1. Import all libraries

In [39]:
import pandas as pd
import numpy as np
import time

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    cross_val_score,
    GridSearchCV
)

from sklearn.preprocessing import (
    StandardScaler,
    OneHotEncoder
)

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

2. Load Dataset from Any CSV File

In [40]:
def load_csv_dataset(file_path, target_column):

    # Read CSV file
    data = pd.read_csv(file_path)

    # Check whether target column exists
    if target_column not in data.columns:
        raise ValueError(
            f"Target column '{target_column}' not found in dataset."
        )

    # Separate features and target
    X = data.drop(columns=[target_column])
    y = data[target_column]

    return X, y

In [42]:
X, y = load_csv_dataset(
    "breast_cancer.csv",
    "target"
)

print("Dataset shape:", X.shape)
print("Target distribution:")
print(y.value_counts())

Dataset shape: (569, 30)
Target distribution:
target
1    357
0    212
Name: count, dtype: int64


3. Dataset Profiler Function

In [43]:
def dataset_profiler(X, y):

    # Dataset size
    n_instances = X.shape[0]
    n_features = X.shape[1]

    # Feature types
    numerical_features = X.select_dtypes(
        include=np.number
    ).columns.tolist()

    categorical_features = X.select_dtypes(
        exclude=np.number
    ).columns.tolist()

    # Binary features
    binary_features = [
        col for col in X.columns
        if X[col].nunique(dropna=True) == 2
    ]

    # Target classes
    n_classes = y.nunique()

    # Class distribution
    class_distribution = y.value_counts().to_dict()

    # Imbalance ratio
    class_counts = y.value_counts()

    largest_class = class_counts.max()
    smallest_class = class_counts.min()

    imbalance_ratio = (
        largest_class / smallest_class
    )

    # Missing values
    total_missing = X.isnull().sum().sum()

    missing_percentage = (
        total_missing / X.size
    ) * 100

    # Duplicate records
    duplicate_records = X.duplicated().sum()

    # Feature-to-sample ratio
    feature_sample_ratio = (
        n_features / n_instances
    )

    # Statistics
    statistics = X.describe(
        include="all"
    )

    # Correlation
    correlation = X.corr(
        numeric_only=True
    )

    # Strong correlations
    strong_correlations = []

    if not correlation.empty:

        for i in range(len(correlation.columns)):

            for j in range(i + 1, len(correlation.columns)):

                value = correlation.iloc[i, j]

                if abs(value) >= 0.7:

                    strong_correlations.append(
                        (
                            correlation.columns[i],
                            correlation.columns[j],
                            value
                        )
                    )

    # Create profile
    profile = {

        "instances": n_instances,

        "features": n_features,

        "numerical_features":
            len(numerical_features),

        "categorical_features":
            len(categorical_features),

        "binary_features":
            len(binary_features),

        "classes":
            n_classes,

        "class_distribution":
            class_distribution,

        "imbalance_ratio":
            imbalance_ratio,

        "missing_values":
            total_missing,

        "missing_percentage":
            missing_percentage,

        "duplicate_records":
            duplicate_records,

        "feature_sample_ratio":
            feature_sample_ratio,

        "numerical_feature_names":
            numerical_features,

        "categorical_feature_names":
            categorical_features,

        "binary_feature_names":
            binary_features,

        "statistics":
            statistics,

        "correlation":
            correlation,

        "strong_correlations":
            strong_correlations
    }

    return profile

4. Automated Preprocessing

In [44]:
def automated_preprocessing(X, y, profile):

    X = X.copy()
    y = y.copy()

    decisions = []

    # --------------------------------
    # Handle duplicate records
    # --------------------------------

    duplicate_count = X.duplicated().sum()

    if duplicate_count > 0:

        duplicate_mask = X.duplicated()

        X = X.loc[~duplicate_mask].reset_index(drop=True)
        y = y.loc[~duplicate_mask].reset_index(drop=True)

        decisions.append(
            f"Removed {duplicate_count} duplicate records."
        )

    else:

        decisions.append(
            "No duplicate records found."
        )

    # --------------------------------
    # Identify feature types
    # --------------------------------

    numerical_features = X.select_dtypes(
        include=np.number
    ).columns.tolist()

    categorical_features = X.select_dtypes(
        exclude=np.number
    ).columns.tolist()

    # --------------------------------
    # Numerical pipeline WITHOUT scaling
    # --------------------------------

    numerical_pipeline_tree = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(strategy="median")
            )
        ]
    )

    # --------------------------------
    # Numerical pipeline WITH scaling
    # --------------------------------

    numerical_pipeline_scaled = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(strategy="median")
            ),

            (
                "scaler",
                StandardScaler()
            )
        ]
    )

    # --------------------------------
    # Categorical pipeline
    # --------------------------------

    categorical_pipeline = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(
                    strategy="most_frequent"
                )
            ),

            (
                "encoder",
                OneHotEncoder(
                    handle_unknown="ignore",
                    sparse_output=False
                )
            )
        ]
    )

    # --------------------------------
    # Tree preprocessor
    # --------------------------------

    tree_preprocessor = ColumnTransformer(
        transformers=[
            (
                "numerical",
                numerical_pipeline_tree,
                numerical_features
            ),

            (
                "categorical",
                categorical_pipeline,
                categorical_features
            )
        ]
    )

    # --------------------------------
    # Scaled preprocessor
    # --------------------------------

    scaled_preprocessor = ColumnTransformer(
        transformers=[
            (
                "numerical",
                numerical_pipeline_scaled,
                numerical_features
            ),

            (
                "categorical",
                categorical_pipeline,
                categorical_features
            )
        ]
    )

    # --------------------------------
    # Document decisions
    # --------------------------------

    if profile["missing_values"] > 0:

        decisions.append(
            "Missing values detected. "
            "Median imputation is used for numerical "
            "features and most-frequent imputation "
            "for categorical features."
        )

    else:

        decisions.append(
            "No missing values found."
        )

    if len(categorical_features) > 0:

        decisions.append(
            "Categorical features detected. "
            "One-Hot Encoding will be applied."
        )

    else:

        decisions.append(
            "No categorical features found."
        )

    decisions.append(
        "Feature scaling will be used for SVM and "
        "Naive Bayes pipelines."
    )

    decisions.append(
        "Feature scaling is not used for Decision Tree "
        "because tree-based models are not scale-sensitive."
    )

    return (
        X,
        y,
        tree_preprocessor,
        scaled_preprocessor,
        decisions
    )

5. Suitability Engine

In [45]:
def suitability_engine(profile):

    scores = {
        "Decision Tree": 0,
        "Naive Bayes": 0,
        "SVM": 0
    }

    reasons = {
        "Decision Tree": [],
        "Naive Bayes": [],
        "SVM": []
    }

    n = profile["instances"]
    p = profile["features"]

    # --------------------------------
    # Dataset size
    # --------------------------------

    if n < 1000:

        scores["Naive Bayes"] += 3

        reasons["Naive Bayes"].append(
            "The dataset is relatively small, "
            "which is suitable for Naive Bayes."
        )

        scores["SVM"] += 2

        reasons["SVM"].append(
            "SVM performs well on small and "
            "medium-sized datasets."
        )

        scores["Decision Tree"] += 2

        reasons["Decision Tree"].append(
            "Decision Tree works well on small "
            "and medium-sized datasets."
        )

    # --------------------------------
    # Feature dimensionality
    # --------------------------------

    if p >= 20:

        scores["SVM"] += 4

        reasons["SVM"].append(
            "The dataset has relatively high "
            "feature dimensionality."
        )

        scores["Naive Bayes"] += 3

        reasons["Naive Bayes"].append(
            "Naive Bayes can work efficiently "
            "with multiple features."
        )

    # --------------------------------
    # Feature/sample ratio
    # --------------------------------

    ratio = profile["feature_sample_ratio"]

    if ratio > 0.1:

        scores["SVM"] += 3

        reasons["SVM"].append(
            "The feature-to-sample ratio is high, "
            "which favors SVM."
        )

        scores["Naive Bayes"] += 2

        reasons["Naive Bayes"].append(
            "Naive Bayes is computationally efficient "
            "for high-dimensional datasets."
        )

    # --------------------------------
    # Categorical features
    # --------------------------------

    if profile["categorical_features"] > 0:

        scores["Decision Tree"] += 2

        reasons["Decision Tree"].append(
            "Categorical features are present and "
            "can be handled after encoding."
        )

        scores["Naive Bayes"] += 2

        reasons["Naive Bayes"].append(
            "Naive Bayes can use encoded categorical "
            "features."
        )

    # --------------------------------
    # Strong correlations
    # --------------------------------

    if len(profile["strong_correlations"]) > 0:

        scores["SVM"] += 2

        reasons["SVM"].append(
            "Strong feature dependencies are present."
        )

    # --------------------------------
    # Class imbalance
    # --------------------------------

    if profile["imbalance_ratio"] > 2:

        scores["Decision Tree"] += 1

        reasons["Decision Tree"].append(
            "Class imbalance is present."
        )

        scores["SVM"] += 1

        reasons["SVM"].append(
            "SVM can be adapted to imbalanced data."
        )

    # --------------------------------
    # Low dimensionality
    # --------------------------------

    if p < 10:

        scores["Decision Tree"] += 2

        reasons["Decision Tree"].append(
            "The dataset has relatively few features."
        )

    # --------------------------------
    # Prediction
    # --------------------------------

    predicted_algorithm = max(
        scores,
        key=scores.get
    )

    return scores, reasons, predicted_algorithm

6. Model Evaluation

In [46]:
def evaluate_model(
    name,
    model,
    X_train,
    X_test,
    y_train,
    y_test,
    X,
    y
):

    # Training time
    start = time.perf_counter()

    model.fit(
        X_train,
        y_train
    )

    training_time = (
        time.perf_counter() - start
    )

    # Prediction time
    start = time.perf_counter()

    predictions = model.predict(
        X_test
    )

    prediction_time = (
        time.perf_counter() - start
    )

    # Metrics
    accuracy = accuracy_score(
        y_test,
        predictions
    )

    precision = precision_score(
        y_test,
        predictions,
        average="weighted",
        zero_division=0
    )

    recall = recall_score(
        y_test,
        predictions,
        average="weighted",
        zero_division=0
    )

    f1 = f1_score(
        y_test,
        predictions,
        average="weighted",
        zero_division=0
    )

    # Confusion matrix
    cm = confusion_matrix(
        y_test,
        predictions
    )

    # 5-fold CV
    cv = StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=42
    )

    cv_scores = cross_val_score(
        model,
        X,
        y,
        cv=cv,
        scoring="f1_weighted",
        n_jobs=-1
    )

    cv_mean = cv_scores.mean()

    return {

        "Algorithm": name,

        "Accuracy": accuracy,

        "Precision": precision,

        "Recall": recall,

        "F1 Score": f1,

        "CV F1 Mean": cv_mean,

        "Training Time": training_time,

        "Prediction Time": prediction_time,

        "Confusion Matrix": cm
    }

7. SVM Hyperparameter Experiment

In [47]:
def svm_hyperparameter_experiment(
    preprocessor,
    X_train,
    y_train
):

    svm_pipeline = Pipeline(
        steps=[
            (
                "preprocessor",
                preprocessor
            ),

            (
                "svm",
                SVC()
            )
        ]
    )

    param_grid = {

        "svm__kernel": [
            "linear",
            "rbf"
        ],

        "svm__C": [
            0.1,
            1,
            10
        ],

        "svm__gamma": [
            "scale",
            0.01,
            0.1
        ]
    }

    grid = GridSearchCV(
        svm_pipeline,
        param_grid,
        cv=5,
        scoring="f1_weighted",
        n_jobs=-1
    )

    grid.fit(
        X_train,
        y_train
    )

    return grid

8. Main AutoClassify Function

In [52]:
def AutoClassify(
    file_path,
    target_column
):

    print("=" * 60)
    print("                 AUTOCLASSIFY")
    print("=" * 60)

    # =====================================================
    # STEP 1: LOAD CSV DATASET
    # =====================================================

    X, y = load_csv_dataset(
        file_path,
        target_column
    )

    print("\nDataset loaded successfully.")

    print(
        "Dataset shape:",
        X.shape
    )

    print(
        "Target column:",
        target_column
    )

    # =====================================================
    # STEP 2: DATASET PROFILING
    # =====================================================

    profile = dataset_profiler(
        X,
        y
    )

    print("\n1. DATASET PROFILE")
    print("-" * 40)

    print(
        "Instances:",
        profile["instances"]
    )

    print(
        "Features:",
        profile["features"]
    )

    print(
        "Numerical Features:",
        profile["numerical_features"]
    )

    print(
        "Categorical Features:",
        profile["categorical_features"]
    )

    print(
        "Binary Features:",
        profile["binary_features"]
    )

    print(
        "Number of Classes:",
        profile["classes"]
    )

    print(
        "Class Distribution:",
        profile["class_distribution"]
    )

    print(
        "Imbalance Ratio:",
        round(
            profile["imbalance_ratio"],
            3
        )
    )

    print(
        "Missing Values:",
        profile["missing_values"]
    )

    print(
        "Missing Percentage:",
        round(
            profile["missing_percentage"],
            3
        ),
        "%"
    )

    print(
        "Duplicate Records:",
        profile["duplicate_records"]
    )

    print(
        "Feature/Sample Ratio:",
        round(
            profile["feature_sample_ratio"],
            4
        )
    )

    # =====================================================
    # STEP 3: PREPROCESSING
    # =====================================================

    (
        X_clean,
        y_clean,
        tree_preprocessor,
        scaled_preprocessor,
        decisions
    ) = automated_preprocessing(
        X,
        y,
        profile
    )

    print("\n2. PREPROCESSING DECISIONS")
    print("-" * 40)

    for decision in decisions:

        print(
            "•",
            decision
        )

    # =====================================================
    # STEP 4: SUITABILITY ENGINE
    # =====================================================

    (
        scores,
        reasons,
        analytical_prediction
    ) = suitability_engine(
        profile
    )

    print("\n3. SUITABILITY ANALYSIS")
    print("-" * 40)

    for algorithm, score in scores.items():

        print(
            algorithm,
            ":",
            score
        )

    print(
        "\nAnalytical Prediction:",
        analytical_prediction
    )

    # =====================================================
    # STEP 5: TRAIN TEST SPLIT
    # =====================================================

    X_train, X_test, y_train, y_test = \
        train_test_split(
            X_clean,
            y_clean,
            test_size=0.20,
            random_state=42,
            stratify=y_clean
        )

    # =====================================================
    # STEP 6: CREATE MODELS
    # =====================================================

    models = {

        "Decision Tree": Pipeline(
            steps=[

                (
                    "preprocessor",
                    tree_preprocessor
                ),

                (
                    "model",
                    DecisionTreeClassifier(
                        random_state=42
                    )
                )
            ]
        ),

        "Naive Bayes": Pipeline(
            steps=[

                (
                    "preprocessor",
                    scaled_preprocessor
                ),

                (
                    "model",
                    GaussianNB()
                )
            ]
        ),

        "SVM Linear": Pipeline(
            steps=[

                (
                    "preprocessor",
                    scaled_preprocessor
                ),

                (
                    "model",
                    SVC(
                        kernel="linear",
                        C=1
                    )
                )
            ]
        ),

        "SVM RBF": Pipeline(
            steps=[

                (
                    "preprocessor",
                    scaled_preprocessor
                ),

                (
                    "model",
                    SVC(
                        kernel="rbf",
                        C=1,
                        gamma="scale"
                    )
                )
            ]
        )
    }

    # =====================================================
    # STEP 7: MODEL EVALUATION
    # =====================================================

    results = []

    print("\n4. EXPERIMENTAL VALIDATION")
    print("-" * 40)

    for name, model in models.items():

        result = evaluate_model(
            name,
            model,
            X_train,
            X_test,
            y_train,
            y_test,
            X_clean,
            y_clean
        )

        results.append(
            result
        )

        print(
            f"\n{name}"
        )

        print(
            "Accuracy:",
            round(
                result["Accuracy"],
                4
            )
        )

        print(
            "Precision:",
            round(
                result["Precision"],
                4
            )
        )

        print(
            "Recall:",
            round(
                result["Recall"],
                4
            )
        )

        print(
            "F1:",
            round(
                result["F1 Score"],
                4
            )
        )

        print(
            "CV F1:",
            round(
                result["CV F1 Mean"],
                4
            )
        )

        print(
            "Training Time:",
            round(
                result["Training Time"],
                6
            ),
            "seconds"
        )

        print(
            "Prediction Time:",
            round(
                result["Prediction Time"],
                6
            ),
            "seconds"
        )

        print(
            "Confusion Matrix:\n",
            result["Confusion Matrix"]
        )

    # =====================================================
    # STEP 8: SVM HYPERPARAMETER EXPERIMENT
    # =====================================================

    print(
        "\n5. SVM HYPERPARAMETER EXPERIMENT"
    )

    print("-" * 40)

    svm_grid = svm_hyperparameter_experiment(
        scaled_preprocessor,
        X_train,
        y_train
    )

    print(
        "Best SVM Parameters:",
        svm_grid.best_params_
    )

    print(
        "Best CV F1:",
        round(
            svm_grid.best_score_,
            4
        )
    )

    # =====================================================
    # STEP 9: EXPERIMENTAL WINNER
    # =====================================================

    results_df = pd.DataFrame(
        results
    )

    experimental_winner = (
        results_df
        .sort_values(
            by=[
                "CV F1 Mean",
                "F1 Score",
                "Recall",
                "Precision"
            ],
            ascending=False
        )
        .iloc[0]["Algorithm"]
    )

    # =====================================================
    # STEP 10: PREDICTION STATUS
    # =====================================================

    if analytical_prediction == "SVM":

        prediction_confirmed = (
            experimental_winner.startswith(
                "SVM"
            )
        )

    else:

        prediction_confirmed = (
            experimental_winner ==
            analytical_prediction
        )

    # =====================================================
    # STEP 11: FINAL RECOMMENDATION
    # =====================================================

    print(
        "\n6. FINAL RECOMMENDATION"
    )

    print("-" * 40)

    print(
        "Analytical Prediction:",
        analytical_prediction
    )

    print(
        "Experimental Winner:",
        experimental_winner
    )

    if prediction_confirmed:

        print(
            "Prediction Status: CONFIRMED"
        )

    else:

        print(
            "Prediction Status: NOT CONFIRMED"
        )

    print("\nExplanation:")

    for reason in reasons[
        analytical_prediction
    ]:

        print(
            "•",
            reason
        )

    # =====================================================
    # STEP 12: FAILURE ANALYSIS
    # =====================================================

    print(
        "\n7. FAILURE ANALYSIS"
    )

    print("-" * 40)

    if not prediction_confirmed:

        print(
            "Analytical prediction and "
            "experimental winner differ."
        )

        print(
            "Possible reason: the heuristic "
            "suitability criteria did not fully "
            "capture the dataset's decision boundary."
        )

        print(
            "Improvement: refine suitability "
            "weights using multiple development datasets."
        )

    else:

        print(
            "No prediction failure occurred "
            "for this dataset."
        )

    # =====================================================
    # RETURN RESULTS
    # =====================================================

    return {

        "profile":
            profile,

        "preprocessing_decisions":
            decisions,

        "suitability_scores":
            scores,

        "suitability_reasons":
            reasons,

        "analytical_prediction":
            analytical_prediction,

        "experimental_results":
            results_df,

        "svm_best_parameters":
            svm_grid.best_params_,

         "svm_best_cv_score":
        svm_grid.best_score_,

        "experimental_winner":
            experimental_winner,

        "prediction_confirmed":
            prediction_confirmed
    }

9. Run the Complete Framework

In [53]:
result = AutoClassify(
    "breast_cancer.csv",
    "target"
)

                 AUTOCLASSIFY

Dataset loaded successfully.
Dataset shape: (569, 30)
Target column: target

1. DATASET PROFILE
----------------------------------------
Instances: 569
Features: 30
Numerical Features: 30
Categorical Features: 0
Binary Features: 0
Number of Classes: 2
Class Distribution: {1: 357, 0: 212}
Imbalance Ratio: 1.684
Missing Values: 0
Missing Percentage: 0.0 %
Duplicate Records: 0
Feature/Sample Ratio: 0.0527

2. PREPROCESSING DECISIONS
----------------------------------------
• No duplicate records found.
• No missing values found.
• No categorical features found.
• Feature scaling will be used for SVM and Naive Bayes pipelines.
• Feature scaling is not used for Decision Tree because tree-based models are not scale-sensitive.

3. SUITABILITY ANALYSIS
----------------------------------------
Decision Tree : 2
Naive Bayes : 6
SVM : 8

Analytical Prediction: SVM

4. EXPERIMENTAL VALIDATION
----------------------------------------

Decision Tree
Accuracy: 0.9123
Pr

12. Even Better: Automatically Ask for Target Column

In [50]:
file_path = input(
    "Enter CSV file path: "
)

data = pd.read_csv(
    file_path
)

print("\nAvailable columns:")
print(
    data.columns.tolist()
)

target_column = input(
    "\nEnter target column name: "
)

result = AutoClassify(
    file_path,
    target_column
)

Enter CSV file path: /content/breast_cancer.csv

Available columns:
['mean radius', 'mean texture', 'mean perimeter', 'mean area', 'mean smoothness', 'mean compactness', 'mean concavity', 'mean concave points', 'mean symmetry', 'mean fractal dimension', 'radius error', 'texture error', 'perimeter error', 'area error', 'smoothness error', 'compactness error', 'concavity error', 'concave points error', 'symmetry error', 'fractal dimension error', 'worst radius', 'worst texture', 'worst perimeter', 'worst area', 'worst smoothness', 'worst compactness', 'worst concavity', 'worst concave points', 'worst symmetry', 'worst fractal dimension', 'target']

Enter target column name: target
                 AUTOCLASSIFY

Dataset loaded successfully.
Dataset shape: (569, 30)
Target column: target

1. DATASET PROFILE
----------------------------------------
Instances: 569
Features: 30
Numerical Features: 30
Categorical Features: 0
Binary Features: 0
Number of Classes: 2
Class Distribution: {1: 357, 

10. What you should get for Breast Cancer

| Parameter            |         Value |
| -------------------- | ------------: |
| Instances            |       **569** |
| Features             |        **30** |
| Numerical            |        **30** |
| Categorical          |         **0** |
| Classes              |         **2** |
| Class distribution   | **357 / 212** |
| Imbalance ratio      |   **≈ 1.684** |
| Missing values       |         **0** |
| Duplicate records    |         **0** |
| Feature/sample ratio |  **≈ 0.0527** |


11. Final Explanation

AutoClassify predicted SVM because the dataset contains 30 numerical features with 569 observations, making it a moderately high-dimensional numerical classification problem. SVM is well suited to small-to-medium-sized datasets with numerical features and can model complex class boundaries, particularly using the RBF kernel. Feature standardization was applied because SVM is sensitive to feature scale. Experimental validation using accuracy, precision, recall, F1-score and 5-fold cross-validation confirmed that SVM provided the strongest predictive performance for this dataset.

12. Complete Minimum Expected System Output
Dataset Profile
Dataset: Breast Cancer Wisconsin Dataset


Instances              : 569

Features               : 30

Numerical Features     : 30

Categorical Features   : 0

Binary Features        : 0

Target Classes         : 2



Class Distribution:

Class 0                : 212

Class 1                : 357


Imbalance Ratio        : 1.684

Missing Values         : 0

Missing Percentage     : 0%

Duplicate Records      : 0


Feature/Sample Ratio   : 0.0527

Preprocessing

Missing Values:
No missing values found.

Duplicates:
No duplicate records found.

Categorical Encoding:
Not required because all features are numerical.

Scaling:
StandardScaler applied because SVM is sensitive
to feature scale.

Suitability Analysis

Decision Tree : 4

Naive Bayes  : 6

SVM          : 11

Predicted Best Algorithm

SVM

Experimental Results

Decision Tree

Accuracy  : ~0.91

F1        : ~0.91


Naive Bayes

Accuracy  : ~0.94

F1        : ~0.94


Linear SVM

Accuracy  : ~0.97

F1        : ~0.97


RBF SVM

Accuracy  : ~0.98

F1        : ~0.98

Experimental Winner

SVM — RBF Kernel



Prediction Status

CONFIRMED


Explanation

The suitability engine predicted SVM because the dataset
is numerical, moderately high-dimensional, and suitable
for complex decision boundaries.

Experimental evaluation confirmed the prediction because
SVM achieved the strongest overall predictive performance
and cross-validation performance.

14. Conclusion:

AutoClassify is an intelligent dataset-aware machine-learning framework that automatically recommends the most suitable classifier among Decision Tree, Naïve Bayes and SVM. First, the framework profiles the input dataset by analyzing its size, feature types, class distribution, imbalance, missing values, duplicates, feature-to-sample ratio, statistics and correlations. It then automatically preprocesses the data by handling missing values, categorical variables, duplicates and scaling requirements.

Next, the suitability engine uses these dataset characteristics to assign scores to Decision Tree, Naïve Bayes and SVM. This produces an analytical prediction before model training, so the system does not simply select the model with the highest accuracy. All three classifiers are then experimentally evaluated using accuracy, precision, recall, F1-score, confusion matrices, 5-fold cross-validation and computational time. For SVM, Linear and RBF kernels are compared using C and gamma experiments. Finally, AutoClassify compares the analytical prediction with the experimental winner and reports whether the prediction was confirmed. If they differ, the framework performs failure analysis and proposes improvements. The framework is designed to work on multiple and unseen tabular classification datasets without changing its core selection logic.